In [ ]:
#Imports , why ??

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
# 1. Hardware setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 2. Data Loading & Transforms for MNIST
# MNIST images are 1x28x28 grayscale. We normalize them using standard dataset mean and std.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [ ]:
# 3. Defining the Custom CNN from Scratch
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Block 1: Input is 1 channel (grayscale), output is 8 channels.
        # Kernel size 3x3, padding 1 keeps the spatial size at 28x28.
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 28x28 -> 14x14
        
        # Block 2: Input 8 channels, output 16 channels.
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 14x14 -> 7x7
        
        # Fully Connected Classification Head
        # After two 2x2 max poolings, 28x28 becomes 7x7.
        # Flattened size = 16 channels * 7 height * 7 width = 784 features.
        self.fc1 = nn.Linear(16 * 7 * 7, 64)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(64, 10)  # 10 output classes for digits 0-9

    def forward(self, x):
        # Pass through Block 1
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        # Pass through Block 2
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        # Flatten feature maps into a 1D vector per batch item
        x = x.view(x.size(0), -1) 
        
        # Pass through Linear Layers
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        
        return x

In [ ]:
model = SimpleCNN().to(device)

In [ ]:
# Print model summary and total parameter count
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {total_params:,}")

In [ ]:
# 4. Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train_model(model, loader, optimizer, criterion, epochs=3):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for batch_idx, (data, target) in enumerate(loader):
            data, target = data.to(device), target.to(device)
            
            # Step A: Clear old gradients
            optimizer.zero_grad()
            
            # Step B: Forward pass (compute predictions)
            output = model(data)
            
            # Step C: Compute loss (how wrong was the model)
            loss = criterion(output, target)
            
            # Step D: Backward pass (compute gradients via backpropagation)
            loss.backward()
            
            # Step E: Update weights
            optimizer.step()
            
            running_loss += loss.item()
            if batch_idx % 300 == 0:
                print(f"Epoch {epoch+1} [{batch_idx * len(data)}/{len(loader.dataset)}] "
                      f"Loss: {loss.item():.4f}")

In [ ]:
# 6. Evaluation Loop
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            
    accuracy = 100. * correct / len(loader.dataset)
    print(f"\nTest Set Accuracy: {accuracy:.2f}%\n")

In [ ]:
# Run training and evaluation
train_model(model, train_loader, optimizer, criterion, epochs=3)
evaluate_model(model, test_loader)

In [ ]:
import torch

# Define a file path to save your weights
model_path = 'simple_cnn_mnist.pth'

# Save the state dictionary (contains all learned weights and biases)
torch.save(model.state_dict(), model_path)
print(f"Model weights successfully saved to {model_path}")

In [ ]:
# 1. Re-instantiate the model architecture
loaded_model = SimpleCNN().to(device)

# 2. Load the saved weights into the model
loaded_model.load_state_dict(torch.load(model_path, map_location=device))

# 3. Switch to evaluation mode (disables Dropout/BatchNorm updates if used)
loaded_model.eval()

print("Model successfully loaded and ready for inference!")

In [ ]:
# 5. Load and preprocess your local image
from PIL import Image
preprocess = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Same normalization used during training
])
image_path = 'D:\\CV\\img_88.jpg' # Replace with your image file path
image = Image.open(image_path).convert('L')

input_tensor = preprocess(image)
# Add a batch dimension: [1, 1, 28, 28] because models expect batches
input_batch = input_tensor.unsqueeze(0).to(device)

In [ ]:
import torch.nn.functional as F
with torch.no_grad():
    raw_logits = model(input_batch)
    
    # Convert raw logits to probabilities using Softmax
    probabilities = F.softmax(raw_logits, dim=1)
    
    # Get the final predicted class
    predicted_digit = raw_logits.argmax(dim=1).item()
    confidence = probabilities[0][predicted_digit].item() * 100

print(f"Raw Logits: {raw_logits.cpu().numpy()}")
print(f"Probabilities per class (0-9): {probabilities.cpu().numpy()}")
print(f"\nPredicted Digit: {predicted_digit} (Confidence: {confidence:.2f}%)")